# Memory Optimization: Garbage Collection Tuning, Off-Heap, and Kryo

**Module**: 10_performance_and_optimization  
**Notebook**: 05_memory_optimization.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Techniques for memory optimization: JVM Garbage Collection diagnostics (G1GC), Kryo serialization, and off-heap memory.
- **Why does this topic exist?**: Memory leaks and GC pauses ruin job predictability and cause heartbeat loss.
- **What problem does it solve?**: Stop-The-World GC pauses exceeding 30 seconds crashing worker heartbeats.
- **Where is it used in real-world Data Engineering?**: Enterprise memory tuning on large JVM clusters.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Memory architecture, serialization, Spark UI |
| **Required depth** | Deep working knowledge: can diagnose GC and memory pressure and choose a safe optimization |
| **Data Engineer** | Reduce object overhead and GC pressure without destabilizing the job. |
| **Senior Data Engineer** | Choose between serialization, off-heap, and data-structure changes based on evidence. |
| **Spark specialist** | Reason about JVM heap layout, GC algorithms, and Spark's binary formats. |
| **Next topic** | `10_performance_and_optimization/06_data_skew_optimization.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Memory optimization is not just "add more RAM." Object overhead, serialization format, caching choices, and
GC pauses often matter more than the raw heap size. Spark's Tungsten binary formats reduce overhead for DataFrames,
but Python objects, UDFs, and user-defined classes can still dominate. Measure GC time and memory before tuning.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Heap** | JVM memory | Stores objects, cached blocks, and execution state |
| **GC** | Garbage collection | Pauses can dominate CPU time |
| **Serialization** | Object-to-bytes | Kryo or Java; affects size and CPU |
| **Off-heap** | Native memory | Avoids some GC pressure but must be sized and monitored |
| **Object overhead** | Per-object cost | Java headers and boxed primitives waste space |
| **Binary rows** | UnsafeRow | Compact representation for DataFrames |

### How it works, step by step

1. The JVM allocates objects for rows, keys, and buffers.
2. GC reclaims unreachable objects; frequent collections increase CPU and pause time.
3. Spark's binary formats reduce per-row overhead for supported operations.
4. Serialization converts objects to bytes for shuffle/cache.
5. Off-heap allocation moves some data outside the GC-managed heap.
6. Tuning changes one of these costs and should be measured against the others.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Memory_Opt_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Inspect serialization configuration
print("Serializer: ", spark.conf.get("spark.serializer"))
print("Is Kryo?:   ", "Kryo" in spark.conf.get("spark.serializer", ""))

---

## Experiment: Kryo Serialization Verification

- **1. What are we testing?**: Verify Kryo is active in our lab environment.
- **2. What do we expect?**: spark.serializer is org.apache.spark.serializer.KryoSerializer.

In [ ]:
print('Active serializer:', spark.sparkContext.getConf().get('spark.serializer'))

**3. What happened?**: KryoSerializer confirmed active.  
**4. Why did it happen?**: Configured in conf/spark-defaults.conf.  
**5. Practical Engineering Lesson**: Kryo is up to 10x faster and significantly more compact than standard Java serialization.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: GC-dominated aggregation

- **Industry**: Advertising
- **Scenario**: A large aggregation spends a significant share of CPU in GC.
- **Problem**: Many short-lived objects from row conversion and string operations create allocation pressure.
- **Why this concept is relevant**: Using native expressions and reducing object churn can lower GC time without increasing heap.
- **Implementation idea**: Compare GC time, allocation rate, and runtime before and after the rewrite.
- **Expected result**: Lower GC overhead and more stable task durations.
- **Alternative**: Increase the heap or switch GC algorithm.
- **Trade-offs**: A larger heap may reduce collection frequency but can increase pause duration.

### Use Case 2: Cache with serialized storage

- **Industry**: Media
- **Scenario**: A cached dataset consumes more memory than expected and evicts other blocks.
- **Problem**: Deserialized Java objects carry significant overhead per row.
- **Why this concept is relevant**: A serialized storage level stores bytes and reduces memory footprint at the cost of deserialization on read.
- **Implementation idea**: Compare cached size, hit ratio, and read latency for MEMORY_ONLY versus MEMORY_AND_DISK_SER.
- **Expected result**: A cache that fits and still provides a benefit.
- **Alternative**: Store the data as a columnar table.
- **Trade-offs**: A table reduces memory pressure but adds I/O and freshness management.

---

## Business Value

- Lower infrastructure cost per job.
- More stable runtime with fewer GC pauses.
- Better cache utilization.
- Evidence-based JVM and serializer settings.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Measure GC time before changing heap or GC settings.
- Prefer native expressions and binary formats over object-heavy UDFs.
- Use Kryo when serialization is a bottleneck, and register classes.
- Size off-heap memory explicitly and monitor it.
- Do not enable off-heap without understanding the total memory budget.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Compare serialized and deserialized cache

**Steps**

1. Cache a dataset with MEMORY_ONLY and record cached size and runtime.
2. Repeat with MEMORY_AND_DISK_SER or a serialized level.
3. Compare memory footprint, hit ratio, and read latency.

**What to observe**: Serialized storage uses less memory but adds deserialization cost on access.

**Healthy result**: The chosen level fits the working set and improves end-to-end runtime.

**Problematic result**: Serialization reduces size but the extra CPU outweighs the benefit.

**Engineering takeaway**: Memory and CPU are traded against each other; measure both.

---

## Trade-offs and Alternatives

### Advantages

- Can reduce CPU and memory
- Multiple tuning levers
- Supported by Spark internals

### Limitations

- Easy to misconfigure
- Off-heap complicates sizing
- GC tuning is workload-specific

### When to use it

- GC-heavy jobs
- Large caches
- Object-heavy pipelines

### When not to use it

- Blind heap increases
- Unregistered Kryo classes
- Off-heap without monitoring

### Alternatives inside Spark

- Kryo
- UnsafeRow
- Off-heap memory
- GC options

### Alternatives outside Spark

- JVM profilers
- Container sizing
- Native libraries

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Off-Heap Setting
**Problem Statement**: Print spark.memory.offHeap.enabled.  
**Input / Context**: spark.conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): GC Pause Detection
**Problem Statement**: Where to look in UI for GC time.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): G1GC Flags
**Problem Statement**: List recommended JVM flags for Java 21.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The off-heap OOM

**Scenario**: A job enables off-heap memory and is killed even though the Java heap looks healthy.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check `spark.memory.offHeap.size` and the container limit.
2. Account for off-heap in the total memory budget.
3. Monitor native memory usage and GC logs.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Off-Heap Setting):
```python
print(spark.conf.get('spark.memory.offHeap.enabled','false'))
```

#### Solution 2 (GC Pause Detection):
```python
print('Check Task GC Time column in Stages tab; red bars in event timeline.')
```

#### Solution 3 (G1GC Flags):
```python
print('-XX:+UseG1GC -XX:InitiatingHeapOccupancyPercent=35')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Kryo Serializer**: Always enable Kryo for faster network and storage serialization.
- **GC Tuning**: Use G1GC to keep pauses below 1 second.

### Essential Technical Terms
- `KryoSerializer`: High performance binary serializer.
- `Stop-The-World`: JVM GC pause freezing all application threads.

### Common Engineering Mistakes
- **Using default Java serialization**: Bulky serialized bytes, slow network transfer.

### Interview Questions on This Topic
- **Q: Why is KryoSerializer preferred over standard Java serialization?**
  - *A*: Kryo produces much smaller binary payloads and serializes objects significantly faster, reducing shuffle write time and network transfer volume.


### More Interview Questions

- **Q: How can serialization reduce memory usage?**
  - *A*: Serialized storage represents rows as bytes instead of Java objects, avoiding per-object headers and references, at the cost of deserialization when the data is read.
- **Q: Why can a larger heap make GC worse?**
  - *A*: A larger heap can reduce collection frequency but increase the duration of full collections and the memory footprint, sometimes worsening pause time.

### What to Study Next
Open 06_data_skew_optimization.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')